# EXPLORA 08 — Del DataFrame al grafico

Practica guiada. El profesor va por el `_Caso`, tu por el `_Base`.

La idea de la sesion: **un grafico responde una pregunta.** Si no sabes que pregunta
responde, no lo hagas.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110

ventas = pd.read_csv("../../_RECURSOS/datasets/ventas.csv",
                     sep=";", decimal=",", encoding="utf-8")

## 0 — Preparar

Graficar datos sucios da graficos limpios con numeros falsos. Siempre se limpia primero.

In [ ]:
ventas = ventas.drop_duplicates().reset_index(drop=True)
ventas["canal"] = ventas["canal"].str.strip().str.title()
ventas["fecha"] = pd.to_datetime(ventas["fecha"], format="mixed", dayfirst=True)
ventas["importe"] = ventas["importe"].fillna(
    (ventas["cantidad"] * ventas["precio_unitario"]
     * (1 - ventas["descuento_pct"] / 100)).round(2))
ventas = ventas[ventas["cantidad"] > 0].copy()
ventas["mes"] = ventas["fecha"].dt.to_period("M").dt.to_timestamp()

print(ventas.shape)

## 1 — Anatomia: `figure` y `axes`

- **`figure`** = el lienzo. Tiene tamano y se guarda a disco.
- **`axes`** = un grafico dentro del lienzo. Tiene titulo, ejes, datos.

Todo lo que hagas, hazlo sobre el `ax`. Es lo que te deja poner varios graficos juntos
mas adelante sin reescribir nada.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot([1, 2, 3, 4], [10, 25, 18, 30], marker="o")
ax.set_title("Esto es un axes dentro de una figure")
ax.set_xlabel("Eje X")
ax.set_ylabel("Eje Y")
plt.tight_layout()
plt.show()

> Vas a ver mucho `plt.plot()` suelto en internet. Funciona, pero dibuja sobre un
> "grafico actual" implicito y se vuelve un lio en cuanto hay mas de uno.
> **Usa siempre `fig, ax = plt.subplots()`.**

## 2 — Barras: comparar categorias

In [ ]:
por_ciudad = ventas.groupby("ciudad")["importe"].sum().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(por_ciudad.index, por_ciudad.values, color="#4c72b0")
ax.set_title("Lima concentra la mayor facturacion", weight="bold")
ax.set_ylabel("Importe (S/)")
ax.yaxis.set_major_formatter(mticker.StrMethodFormatter("{x:,.0f}"))
ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()

Dos cosas que casi nadie hace y cambian todo:

1. **Ordenar.** Un grafico de barras sin ordenar obliga al lector a hacer el ranking mentalmente.
2. **Titulo con el hallazgo**, no con el nombre de la variable. "Ventas por ciudad" no dice nada
   que el eje no diga ya.

## 3 — Linea: evolucion en el tiempo

Linea **solo** para tiempo o para algo continuo. Unir con una linea categorias sueltas
(ciudades, productos) sugiere una continuidad que no existe.

In [ ]:
serie = ventas.groupby("mes")["importe"].sum()

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.plot(serie.index, serie.values, marker="o", linewidth=2)
ax.fill_between(serie.index, serie.values, alpha=0.12)
ax.set_title("Facturacion mensual 2025", weight="bold")
ax.set_ylabel("Importe (S/)")
ax.yaxis.set_major_formatter(mticker.StrMethodFormatter("{x:,.0f}"))
fig.autofmt_xdate(rotation=45)
plt.tight_layout()
plt.show()

## 4 — Histograma: distribucion

Barras comparan **categorias**. El histograma muestra **como se reparte una variable numerica**.
No son lo mismo aunque se parezcan.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))

axes[0].hist(ventas["importe"], bins=30, color="#4c72b0", edgecolor="white")
axes[0].set_title("Escala lineal: todo aplastado a la izquierda")
axes[0].set_xlabel("Importe (S/)")

axes[1].hist(ventas["importe"], bins=30, color="#c8102e", edgecolor="white")
axes[1].set_xscale("log")
axes[1].set_title("Escala log: ahora si se ve la forma")
axes[1].set_xlabel("Importe (S/, log)")

plt.tight_layout()
plt.show()

La cola larga es tipica de datos de ventas. Consecuencia practica:
**el promedio no representa al cliente tipico.** Usa la mediana.

In [ ]:
print(f"Promedio : S/ {ventas['importe'].mean():>10,.2f}")
print(f"Mediana  : S/ {ventas['importe'].median():>10,.2f}")

## 5 — El atajo de pandas

Para explorar rapido, `df.plot()` ahorra codigo. Para el grafico final, matplotlib directo
te da el control que necesitas.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
(ventas.groupby("canal")["importe"].sum()
 .sort_values()
 .plot(kind="barh", ax=ax, color="#55a868"))
ax.set_title("Facturacion por canal", weight="bold")
ax.set_xlabel("Importe (S/)")
ax.xaxis.set_major_formatter(mticker.StrMethodFormatter("{x:,.0f}"))
plt.tight_layout()
plt.show()

## 6 — Seaborn: boxplot

Seaborn trabaja directo sobre el DataFrame: le pasas `data`, `x`, `y` y el agrupa solo.

La caja va del percentil 25 al 75, la linea del medio es la **mediana**, y los puntos
sueltos son valores atipicos.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.boxplot(data=ventas, x="canal", y="importe", ax=ax)
ax.set_yscale("log")
ax.set_title("Distribucion del ticket por canal", weight="bold")
ax.set_ylabel("Importe (S/, log)")
plt.tight_layout()
plt.show()

## 7 — Seaborn: heatmap

Para cruces de dos categorias. `annot=True` siempre: sin numeros, el lector tiene que
estimar por intensidad de color, y nadie estima bien por color.

In [ ]:
tabla = ventas.pivot_table(index="ciudad", columns="canal",
                           values="importe", aggfunc="sum", fill_value=0)

fig, ax = plt.subplots(figsize=(7, 4))
sns.heatmap(tabla, annot=True, fmt=",.0f", cmap="Blues", linewidths=.5, ax=ax)
ax.set_title("Importe por ciudad y canal", weight="bold")
plt.tight_layout()
plt.show()

## 8 — Subplots 2x2

Aqui se paga el haber usado `ax` desde el principio: es copiar y pegar, cambiando el `ax`.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8))

axes[0, 0].bar(por_ciudad.index, por_ciudad.values, color="#4c72b0")
axes[0, 0].set_title("Por ciudad", weight="bold")
axes[0, 0].tick_params(axis="x", rotation=30)

axes[0, 1].plot(serie.index, serie.values, marker="o")
axes[0, 1].set_title("Evolucion mensual", weight="bold")
axes[0, 1].tick_params(axis="x", rotation=45)

sns.boxplot(data=ventas, x="canal", y="importe", ax=axes[1, 0])
axes[1, 0].set_yscale("log")
axes[1, 0].set_title("Ticket por canal", weight="bold")

sns.heatmap(tabla, annot=True, fmt=",.0f", cmap="Blues", cbar=False, ax=axes[1, 1])
axes[1, 1].set_title("Ciudad x Canal", weight="bold")

for ax in axes.flat:
    ax.yaxis.set_major_formatter(mticker.StrMethodFormatter("{x:,.0f}"))

fig.suptitle("Ventas 2025 — panorama", fontsize=14, weight="bold")
plt.tight_layout()
plt.show()

## 9 — Guardar

- `dpi=150` para que no se vea pixelado en una pantalla o proyector.
- `bbox_inches="tight"` para que no corte titulos ni etiquetas. Sin esto, el titulo
  de arriba se pierde y no te enteras hasta que lo abres.
- `fig.savefig`, no `plt.savefig`: guardas **esa** figura, no la que este activa.

In [ ]:
fig.savefig("panorama_ventas.png", dpi=150, bbox_inches="tight")

from pathlib import Path
print("Guardado:", round(Path("panorama_ventas.png").stat().st_size / 1024), "KB")

---
### Lo que hay que llevarse

1. **`fig, ax = plt.subplots()` siempre.** Nunca `plt.plot()` suelto.
2. **Un grafico = una pregunta.** Si no sabes cual, no lo hagas.
3. **El titulo dice el hallazgo**, no el nombre de la columna.
4. **Ordenar las barras.**
5. **Escala log** cuando el rango abarca varios ordenes de magnitud.
6. **Formatear el eje** con separador de miles.
7. **`bbox_inches="tight"`** al guardar.
8. **Nada de pastel** con mas de 3 categorias.